# TP 4.1 : venv, dépendances et variables d'environnement

## Objectif

`fleetcheck` doit s'installer partout avec la version de `requests` testée, et recevoir son secret sans qu'il passe par le dépôt :

1. point de départ : vérifier que le kernel tourne dans le venv, puis récupérer la version de départ du chapitre 4 ;
2. installer `requests` et figer la dépendance dans `requirements.txt` ;
3. lire l'URL du webhook d'alerte dans l'environnement, sans jamais l'afficher.

## Mode d'emploi du notebook

- Copiez ce notebook à la racine de votre dépôt, à côté de `fleetcheck.py`. Le code s'écrit dans `fleetcheck.py`, ouvert dans un onglet de JupyterLab ; le notebook sert à essayer et vérifier.
- Exécutez d'abord la cellule d'outils ci-dessous. Chaque vérification affiche `OK`, ou `KO` avec ce qui était attendu.
- `%run -n fleetcheck.py` charge les fonctions du fichier sans lancer `main()`. N'appelez jamais `main()` dans le notebook : argparse y lirait les arguments du kernel.
- `run(...)` lance le script comme dans un terminal et renvoie `r.stdout`, `r.stderr` et `r.returncode`. Le script hérite des variables d'environnement du kernel (étape 3). `quiet=True` masque stdout.
- Les vérifications remettent en place ce qu'elles modifient : ne les modifiez pas.
- **Prédire** et **Justifier** : répondez en une ou deux lignes dans la cellule `# Ma réponse :`, **avant** d'exécuter la suite.
- Rendu : *Kernel > Restart Kernel and Run All Cells*, puis commit et push de `fleetcheck.py`, `config.json`, `requirements.txt`, `servers.csv`, `servers-broken.csv` et de ce notebook.

In [ ]:
# Outils du TP : exécuter cette cellule en premier, sans la modifier.
import os
import subprocess
import sys


def check(obtenu, attendu):
    if obtenu == attendu:
        print("OK")
    elif isinstance(attendu, str) and "\n" in attendu:
        print(f"KO, obtenu :\n{obtenu}\nattendu :\n{attendu}")
    else:
        print(f"KO\n  obtenu  : {obtenu!r}\n  attendu : {attendu!r}")


def check_error(fonction, *args, attendu):
    try:
        fonction(*args)
    except Exception as e:
        check(f"{type(e).__name__}: {e}", attendu)
    else:
        print(f"KO, aucune exception levée. Attendu : {attendu}")


def run(*args, quiet=False):
    """Lance fleetcheck.py comme dans un terminal (vu au TP 4.3 : subprocess). quiet=True n'affiche pas stdout."""
    r = subprocess.run([sys.executable, "fleetcheck.py", *args], capture_output=True,
                       encoding="utf-8", env={**os.environ, "PYTHONUTF8": "1", "NO_COLOR": "1",
                                              "PYTHONWARNDEFAULTENCODING": "1"})
    if not quiet:
        print(r.stdout, end="", flush=True)
    print(r.stderr, end="", file=sys.stderr, flush=True)
    print(f"[code de sortie : {r.returncode}]")
    if "EncodingWarning" in r.stderr:
        print("KO : un open(), read_text() ou subprocess text=True sans encoding= (ligne indiquée dans l'EncodingWarning ci-dessus)")
    return r

## Point de départ (étape 1)

### Le venv du kernel

Le kernel doit être le Python 3.14 de `esgi-python/.venv`, créé au TP 0.1, hors de votre dépôt.

- `sys.executable` : l'interpréteur qui exécute le kernel ;
- `sys.prefix` : le dossier de l'environnement (`…/esgi-python/.venv`) ;
- `sys.base_prefix` : le Python d'origine, installé depuis python.org.

**Prédire** — Que vaut `sys.prefix == sys.base_prefix` dans ce notebook ?

In [ ]:
# Ma réponse :

In [ ]:
from pathlib import Path

print(sys.executable)
print(sys.prefix)
print(sys.base_prefix)
check(sys.version_info[:2], (3, 14))
check(sys.prefix != sys.base_prefix, True)
check(Path(sys.prefix).resolve().is_relative_to(Path.cwd().resolve()), False)

Les trois `OK` : passez à « La version de départ ». Le troisième dit que le venv est hors du dépôt : git ne le voit pas. Un venv créé dans un dépôt se met dans le `.gitignore` : il se recrée, il ne se commite pas.

Un `KO` :

- seul `sys.prefix != sys.base_prefix` est `KO` : JupyterLab a été lancé hors du venv. Fermez-le (Ctrl+C dans son terminal), puis, dans un terminal ouvert dans `esgi-python` : `.venv/bin/python -m jupyter lab` (Windows : `.venv\Scripts\python -m jupyter lab`) ;
- la version est `KO`, ou `.venv/bin/python` (Windows : `.venv\Scripts\python`) est introuvable (autre machine, venv supprimé ou copié) : recréez le venv. Fermez JupyterLab, puis, dans un terminal ouvert dans `esgi-python` :

macOS, Linux :

```text
python3.14 -m venv --clear .venv
.venv/bin/python -m pip install --upgrade pip
.venv/bin/python -m pip install jupyterlab
.venv/bin/python -m jupyter lab
```

Windows (PowerShell) :

```text
py -3.14 -m venv --clear .venv
.venv\Scripts\python -m pip install --upgrade pip
.venv\Scripts\python -m pip install jupyterlab
.venv\Scripts\python -m jupyter lab
```

Rouvrez ce notebook et relancez la cellule d'outils puis celle-ci. Machine neuve, sans Python 3.14 ni dépôts : `tp/tp0.1-installation.md`, sections 1 et 4.

### La version de départ

Tout le monde repart de la référence au tag `tp2.2` du dépôt public : elle remplace votre `fleetcheck.py` du TP 2.2, qui reste dans votre historique git.

**Prédire** — `servers-broken.csv` contient 3 lignes invalides. Quel code de sortie pour `run("--inventory", "servers-broken.csv")` ?

In [ ]:
# Ma réponse :

Dans le terminal ouvert depuis le lanceur (dossier de votre dépôt) :

```text
git -C ../fleetcheck pull
git -C ../fleetcheck fetch --tags --force
git -C ../fleetcheck switch --detach tp2.2
cp ../fleetcheck/reference/fleetcheck.py ../fleetcheck/reference/config.json ../fleetcheck/reference/*.csv .
git -C ../fleetcheck switch main
git add fleetcheck.py config.json servers.csv servers-broken.csv
git commit -m "Version de départ du chapitre 4 (tag tp2.2)"
```

- PowerShell : la ligne `cp` devient `Copy-Item ..\fleetcheck\reference\fleetcheck.py, ..\fleetcheck\reference\config.json, ..\fleetcheck\reference\*.csv .`.
- `git pull` refuse à cause de modifications locales : `git -C ../fleetcheck restore .`, puis relancez-le.
- `switch --detach tp2.2` affiche `HEAD est maintenant sur … tp3.1` : `tp3.1` est le message du commit. Vous êtes bien sur `tp2.2`.

Ce que la version de départ apporte (chapitre 3) :

- options `--inventory` (défaut `servers.csv`), `--env`, `--format text|json`, `-v` (log `INFO`), `-vv` (`DEBUG`) ;
- log par `logging` sur stderr, format `NIVEAU message` : un `WARNING ligne N ignorée : …` par ligne invalide, un `ERROR` par erreur fatale ; stdout ne porte que le résultat ;
- avec `-v` : `INFO 9 serveur(s) valide(s), 0 ligne(s) invalide(s) dans servers.csv`, puis `INFO rapport écrit : report.json` ;
- code de sortie : `exit_code(statuses, invalid_count)` renvoie `2` s'il y a une ligne invalide, `1` si un statut vaut `"down"`, `0` sinon ; une erreur fatale sort en `2`.

Vérifiez avant de commencer :

In [ ]:
RESUME = """9 serveurs valides, 0 invalides
par env : prod 7, staging 2
par rôle : web 3, api 3, worker 1, cache 1, db 1
prod/web : web-1, web-2
"""

os.environ.pop("FLEETCHECK_WEBHOOK_URL", None)
r = run()
check((r.stdout, r.stderr, r.returncode), (RESUME, "", 0))

In [ ]:
r = run("--inventory", "servers-broken.csv")
r.returncode

## Étapes

### 2. `requests` et `requirements.txt`

À produire : `requests` 2.34.2 dans le kernel, et un `requirements.txt` d'une ligne.

`%pip` installe dans le Python du kernel. JupyterLab dépend déjà de `requests` : `%pip` répond « already satisfied » ou remplace la version présente. Sans réseau, l'enseignant donne la commande de repli.

In [ ]:
%pip install requests==2.34.2

Écrivez `requirements.txt` à la racine du dépôt : une ligne par dépendance que le projet importe, version figée par `==`.

```text
requests==2.34.2
```

**Justifier** — Pourquoi `requests==2.34.2` plutôt que `requests` seul ?

In [ ]:
# Ma réponse :

Vérification : la bonne version s'importe, `requirements.txt` contient `requests==2.34.2` et pas les dépendances de `requests` (`urllib3`, `idna`…, que `pip freeze` afficherait).

In [ ]:
import re
import requests

check(requests.__version__, "2.34.2")
check(Path("requirements.txt").exists(), True)
if Path("requirements.txt").exists():
    lignes = [l.strip() for l in Path("requirements.txt").read_text(encoding="utf-8").splitlines() if l.strip()]
    check(all(re.fullmatch(r"[A-Za-z0-9._-]+==[0-9][A-Za-z0-9.]*", l) for l in lignes), True)
    check("requests==2.34.2" in lignes, True)
    check([l for l in lignes if l.split("==")[0].lower().replace("_", "-") in {"urllib3", "idna", "certifi", "charset-normalizer"}], [])

Un collègue ou la CI reconstruit l'environnement par `python -m pip install -r requirements.txt`.

### 3. `FLEETCHECK_WEBHOOK_URL`

Au TP 5.2, `fleetcheck` postera une alerte vers un webhook. Son URL est un secret : quiconque la connaît poste dans le salon.

À produire, dans `main()`, après le chargement de l'inventaire et de la config :

- `os.environ.get("FLEETCHECK_WEBHOOK_URL")`, avec `import os` en tête du fichier ;
- présente : log `INFO FLEETCHECK_WEBHOOK_URL défini : alerte activée` ;
- absente : log `INFO FLEETCHECK_WEBHOOK_URL absent : pas d'alerte` ;
- jamais la valeur, ni dans le log ni sur stdout : le log de CI est lisible par toute l'équipe.

Log attendu avec `-v`, variable absente :

```text
INFO 9 serveur(s) valide(s), 0 ligne(s) invalide(s) dans servers.csv
INFO FLEETCHECK_WEBHOOK_URL absent : pas d'alerte
INFO rapport écrit : report.json
```

Pour essayer dans le terminal (dossier du dépôt) :

- bash, zsh : `export FLEETCHECK_WEBHOOK_URL=http://127.0.0.1:8000/webhook`, puis `../.venv/bin/python fleetcheck.py -v` ;
- PowerShell : `$env:FLEETCHECK_WEBHOOK_URL = "http://127.0.0.1:8000/webhook"`, puis `..\.venv\Scripts\python fleetcheck.py -v`.

Dans le notebook, `os.environ[...] = ...` suffit : `run(...)` hérite de l'environnement du kernel.

**Justifier** — Pourquoi l'URL du webhook dans l'environnement plutôt que dans `config.json` ?

In [ ]:
# Ma réponse :

Vérification, variable absente :

In [ ]:
os.environ.pop("FLEETCHECK_WEBHOOK_URL", None)
r = run("-v")
check([l for l in r.stderr.splitlines() if "FLEETCHECK_WEBHOOK_URL" in l], ["INFO FLEETCHECK_WEBHOOK_URL absent : pas d'alerte"])
check((r.stdout, r.returncode), (RESUME, 0))

Vérification, variable posée puis retirée : l'URL n'apparaît nulle part.

In [ ]:
URL = "http://127.0.0.1:8000/webhook"
os.environ["FLEETCHECK_WEBHOOK_URL"] = URL
try:
    r = run("-v")
    sans_option = run()
finally:
    del os.environ["FLEETCHECK_WEBHOOK_URL"]
check([l for l in r.stderr.splitlines() if "FLEETCHECK_WEBHOOK_URL" in l], ["INFO FLEETCHECK_WEBHOOK_URL défini : alerte activée"])
check([l for l in (r.stderr + r.stdout).splitlines() if URL in l], [])
check((sans_option.stderr, sans_option.returncode), ("", 0))

## Critère final

Variable absente, les sorties de la version de départ ne changent pas :

| Lancement | stdout | stderr | Code |
|---|---|---|---|
| `run()` | le résumé | rien | `0` |
| `run("--inventory", "servers-broken.csv")` | le résumé | les 3 `WARNING` | `2` |
| `run("--env", "prd")` | rien | `ERROR aucun serveur pour --env prd` | `2` |
| `run("--format", "json")` | un JSON valide | rien | `0` |

In [ ]:
import json

os.environ.pop("FLEETCHECK_WEBHOOK_URL", None)
r = run()
check((r.stdout, r.stderr, r.returncode), (RESUME, "", 0))
r = run("--inventory", "servers-broken.csv")
check((r.stdout.splitlines()[0], r.stderr.count("WARNING ligne"), r.returncode), ("9 serveurs valides, 3 invalides", 3, 2))
r = run("--env", "prd")
check((r.stdout, r.stderr, r.returncode), ("", "ERROR aucun serveur pour --env prd\n", 2))
r = run("--format", "json", quiet=True)
try:
    summary = json.loads(r.stdout)["summary"]
except json.JSONDecodeError as e:
    summary = None
    print(f"KO : stdout n'est pas un JSON seul ({e}). Début de stdout : {r.stdout[:120]!r}")
check((summary, r.stderr, r.returncode), ({"total": 9, "invalid": 0}, "", 0))

## Pour aller plus loin (facultatif, non exigé au rendu)

- `%pip show requests` : `Requires` liste ce que `requests` tire avec lui, `Required-by` ce qui en dépend dans le venv. Pourquoi aucune de ces lignes ne va dans `requirements.txt` ?

In [ ]:
%pip show requests

- Venv neuf dans un dossier temporaire : `python3.14 -m venv /tmp/essai`, puis `/tmp/essai/bin/python -m pip install -r requirements.txt` (Windows : `py -3.14 -m venv $env:TEMP\essai`, puis `& "$env:TEMP\essai\Scripts\python" -m pip install -r requirements.txt`, le `&` est requis devant un chemin qui commence par `$`). `requests` s'importe dans ce venv, sans rien d'autre que ce que liste le fichier.
- `os.environ.get("FLEETCHECK_ATTEMPTS")` renvoie une chaîne, ou `None` si la variable est absente : faites-en un entier, et un `ERROR` qui nomme la variable, en code `2`, si la valeur n'est pas un entier ≥ 1. Absente, elle ne change rien : au TP 5.2, le nombre d'essais viendra de `config["attempts"]`, que la variable remplacera seulement si elle est posée.

Vérification, variable posée à `abc`, puis retirée. Sans `ERROR`, la cellule l'indique sans `KO`.

In [ ]:
os.environ["FLEETCHECK_ATTEMPTS"] = "abc"
try:
    r = run()
finally:
    del os.environ["FLEETCHECK_ATTEMPTS"]
erreurs = [l for l in r.stderr.splitlines() if l.startswith("ERROR ")]
if erreurs:
    check((r.stdout, r.returncode, "FLEETCHECK_ATTEMPTS" in erreurs[0]), ("", 2, True))
else:
    print("Aucun ERROR : extension FLEETCHECK_ATTEMPTS non traitée")

- Un serveur en objet : la classe `Server` de la slide 08 du chapitre 1. `class Server:` est le plan ; `Server(...)` fabrique une instance, qui porte ses propres attributs ; `self` est l'instance sur laquelle la méthode est appelée (`s.is_prod()` exécute `is_prod` avec `self = s`) ; `__init__` est appelée à la création. Écrivez la classe dans la cellule vide ci-dessous, pas dans `fleetcheck.py`, qui garde ses dicts pour les TP suivants.

```text
class Server:
    def __init__(self, name, env, ip, port, role):
        self.name = name
        # à écrire : les quatre autres attributs

    def is_prod(self):
        # à écrire : True si l'environnement vaut "prod"

    def url(self, path):
        # à écrire : f"http://<ip>:<port><path>", l'URL que le TP 5.1 interrogera
```

In [ ]:
# Votre classe Server :

La cellule suivante fait un objet de chaque dict de `load()` : `Server(**s)` passe chaque clé en argument nommé (`name=…`, `env=…`), donc les paramètres de `__init__` portent les noms des colonnes du CSV (sinon `TypeError: … unexpected keyword argument`). Avant de l'exécuter : que va afficher `print(parc[0])` ?

In [ ]:
%run -n fleetcheck.py
servers, invalid = load("servers.csv")
if "Server" in globals():
    parc = [Server(**s) for s in servers]
    print(parc[0])
    print(parc[0].url("/health"), parc[0].is_prod())
else:
    print("Aucune classe Server : extension non traitée")

`<__main__.Server object at 0x…>` : Python donne la classe et l'adresse en mémoire. Ajoutez une méthode `__repr__(self)` qui renvoie `f"Server({self.name!r}, {self.env!r})"`, relancez la cellule de la classe puis la précédente : `print` affiche `Server('web-1', 'prod')`. Les méthodes encadrées de `__` sont appelées par Python lui-même (`__init__` à la création, `__repr__` à l'affichage). Au chapitre 6, `@dataclass` les écrira pour vous.

Vérification. Sans classe `Server`, la cellule l'indique sans `KO`.

In [ ]:
if "Server" in globals():
    parc = [Server(**s) for s in servers]
    check(parc[0].url("/health"), f"http://{servers[0]['ip']}:{servers[0]['port']}/health")
    check(sum(1 for s in parc if s.is_prod()), 7)
    check(repr(parc[0]), f"Server({servers[0]['name']!r}, {servers[0]['env']!r})")
else:
    print("Aucune classe Server : extension non traitée")